# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muzammilsharf/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

- Unit of analysis: One row represents the daily search and web performance for a single specific piece of content (content_hash_id) belonging to a specific client (client_hash_id), sourced from the fact_content_daily_performance table.

- Time window: The entire month of March 2026 (report_date spanning 2026-03-01 to 2026-03-31).

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- Feature: gsc_clicks, gsc_impressions, ga4_sessions, sessions_ai, gsc_avg_position. (Knowable at the decision moment because daily logs are finalized and sealed at midnight).

- Label: is_declining_label. (A proxy target predicting whether gsc_clicks drop significantly in the subsequent 30-day window).

- Context: report_date, client_hash_id, content_hash_id. (Pure identifiers, never passed to the ML model).

- Excluded: Any data from April 2026 onward. Why: Allowing the model to see future performance metrics while training on March data would cause target leakage, giving it the "answer key" before it makes a prediction.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
from huggingface_hub import list_repo_files

print("\nFetching file list...")
all_files = list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")

daily_files = [f for f in all_files if "fact_content_daily_performance" in f]

for f in daily_files:
    print(f)

/home/muzammil/flyrank-ml-internship/venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Fetching file list...
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/month=2026-04/data_0.parquet
fact_content_dail

In [2]:
from huggingface_hub import hf_hub_download
import pandas as pd

march_file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet"
)

df = pd.read_parquet(march_file_path)
print("Columns:", df.columns.tolist())
df.head()

Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# 1. Verify the Grain: Should return 0 if one row = one unique piece of content per client, per day
grain_cols = ['client_hash_id', 'content_hash_id', 'report_date']
grain_violations = df.duplicated(subset=grain_cols).sum()
print(f"1. Grain violations (duplicates): {grain_violations}")

# 2. Verify the Time Window & Counts
print(f"2. Scope: {len(df):,} rows")
print(f"   Date span: {df['report_date'].min()} to {df['report_date'].max()}")

# 3. Verify Availability (Filtering with IS TRUE as required)
# Let's see how many rows actually have Google Search Console data available
gsc_survivors = df[df['gsc_data_available'] == True]
print(f"3. Availability: {len(gsc_survivors):,} rows survive the 'gsc_data_available IS TRUE' filter")

1. Grain violations (duplicates): 0
2. Scope: 9,841,378 rows
   Date span: 2026-03-01 to 2026-03-31
3. Availability: 3,611,061 rows survive the 'gsc_data_available IS TRUE' filter


In [4]:
# --- Five features, each with an "available when?" line ---

features_df = df[df['gsc_data_available'] == True].copy()
features_df['ctr'] = features_df['gsc_clicks'] / features_df['gsc_impressions']

feature_frame = features_df[[
    'content_hash_id', 'report_date',
    'gsc_impressions',   # available: logged and sealed at midnight each day
    'gsc_avg_position',  # available: computed from that day's GSC data at ingestion
    'ctr',               # available: derived from clicks/impressions, same-day
    'ga4_sessions',      # available: GA4 daily export, same-day
    'sessions_ai'        # available: GA4 daily export, same-day
]].dropna()

print(feature_frame.shape)
feature_frame.head()

(2082695, 7)


,content_hash_id,report_date,gsc_impressions,gsc_avg_position,ctr,ga4_sessions,sessions_ai
10005,content_7195ab099203f112,2026-03-01,1,47.000000,0.0,0.0,0.0
10308,content_4a551d6b2bb20a46,2026-03-01,1,4.000000,0.0,0.0,0.0
10507,content_e45cd1313d99c076,2026-03-01,3,6.666667,0.0,0.0,0.0
10803,content_60c47a2bd592c26e,2026-03-01,3,65.333333,0.0,0.0,0.0
10849,content_a6eb2a271fe81e54,2026-03-01,2,8.000000,0.0,0.0,0.0


In [5]:
# --- The trap: deliberate leak, then remove it ---

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

leak_df = features_df.copy()
leak_df['is_declining_label'] = (leak_df.groupby('content_hash_id')['gsc_clicks'].diff() < 0).astype(int)
leak_df = leak_df.dropna(subset=['is_declining_label'])

# DELIBERATE LEAK: trend_direction-equivalent, directly derived from the label itself
leak_df['leaky_feature'] = leak_df['is_declining_label']  # obviously circular, on purpose

honest_features = ['gsc_impressions', 'gsc_avg_position', 'ctr']
leaky_features = honest_features + ['leaky_feature']

X_train, X_test, y_train, y_test = train_test_split(
    leak_df[leaky_features], leak_df['is_declining_label'], test_size=0.25, random_state=42
)

# WITH the leak
rf_leaky = RandomForestClassifier(n_estimators=50, random_state=42).fit(X_train, y_train)
leaky_score = rf_leaky.score(X_test, y_test)
print(f"Accuracy WITH leaked feature: {leaky_score:.3f}")

# WITHOUT the leak (honest)
rf_honest = RandomForestClassifier(n_estimators=50, random_state=42).fit(X_train[honest_features], y_train)
honest_score = rf_honest.score(X_test[honest_features], y_test)
print(f"Accuracy WITHOUT leaked feature (honest): {honest_score:.3f}")

Accuracy WITH leaked feature: 1.000
Accuracy WITHOUT leaked feature (honest): 0.908


With the label-derived feature included, accuracy hits a suspicious 1.000, a clear signal of leakage since no real-world feature should let a model achieve perfect prediction. Removing it drops accuracy to a believable 0.908, confirming the earlier feature was circular (predicting the label from a copy of itself) rather than genuine signal. This is the leakage check the data contract requires: features must be knowable before the decision point, never derived from the outcome they're meant to predict.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- Algorithmic Volatility: This data shows historical Google Search Console and GA4 metrics, but cannot predict sudden search engine core algorithm updates that might instantly tank a previously healthy page.

- Missing Context: I have engagement metrics (like ga4_total_engagement_sec), but this slice doesn't contain the actual text or quality of the content. A page might decline simply because the information on it became outdated, which numeric logs alone cannot flag.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.